Explore the 2018–2025 regular-season data produced by `dataset_creation.ipynb`. Saved outputs have been cleared for publication; rerun all cells to generate results.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window as W
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import matplotlib.pyplot as plt

In [ ]:
spark = (
    SparkSession.builder
    .appName("EDA")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.parquet.columnarReaderBatchSize", "1024")
    .getOrCreate()
)

In [ ]:
throws_df = spark.read.parquet("./data/full_dataset.parquet")
throws_df.show(5)
throws_df.groupBy("game_year").count().orderBy("game_year").show()


In [ ]:
throws_df.printSchema()

In [ ]:
throws_df.count()

In [ ]:
throws_df.select(
    F.min("game_date"),
    F.max("game_date"),
    F.count("*").alias("n_pitches"),
    F.countDistinct("game_pk").alias("n_games"),
    F.countDistinct("pitcher").alias("n_pitchers")
).show()

In [ ]:
throws_df.select([F.sum(F.col(col).isNull().cast("int")).alias(col) for col in throws_df.columns]).show()
# da indagare bene, ci sono sia nulli sensati (da modificare) che dati mancanti (da rimuovere)

In [ ]:
numerical = ["release_speed", "release_pos_x", "release_pos_y", "release_pos_z",
             "effective_speed", "release_spin_rate", "release_extension",
             "spin_axis", "pfx_x", "pfx_z", "plate_x", "plate_z",
             "vx0", "vy0", "vz0", "ax", "ay", "az",
             "api_break_z_with_gravity", "api_break_x_arm",
             "api_break_x_batter_in", "arm_angle"]

correlation_df = throws_df.select(numerical).dropna()
print("Complete-case correlation observations:", correlation_df.count())
assembler = VectorAssembler(
    inputCols=numerical,
    outputCol="features"
)

assembled_df = assembler.transform(correlation_df)
correlation_matrix = Correlation.corr(
    assembled_df,
    "features",
    method="pearson"
).head()[0].toArray()

plt.figure(figsize=(14, 12))
plt.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Pearson correlation")
plt.xticks(range(len(numerical)), numerical, rotation=90)
plt.yticks(range(len(numerical)), numerical)
plt.title("Correlation matrix of numerical pitch features")
plt.tight_layout()
plt.show()

In [ ]:
target = ["pitch_type", "pitch_name"]
ids = ["game_pk", "game_date", "game_year", "game_type", "at_bat_number", "pitch_number", "pitcher", "batter", "player_name", "home_team", "away_team"]
pre_pitch = ["balls", "strikes", "outs_when_up", "inning", "inning_topbot", "on_1b", "on_2b", "on_3b", "home_score", "away_score", "bat_score", "fld_score", "home_score_diff", "bat_score_diff", "n_thruorder_pitcher", "n_priorpa_thisgame_player_at_bat"]
pitcher = ["stand", "p_throws", "age_pit", "age_bat", "pitcher_days_since_prev_game"]
pitch = ["release_speed", "release_pos_x", "release_pos_y", "release_pos_z", "effective_speed", "release_spin_rate", "release_extension", "spin_axis", "arm_angle", "pfx_x", "pfx_z", "plate_x", "plate_z", "vx0", "vy0", "vz0", "ax", "ay", "az", "api_break_z_with_gravity", "api_break_x_arm", "api_break_x_batter_in"]
result = ["description", "type", "zone", "events", "des"]
batted = ["hit_location", "bb_type", "hc_x", "hc_y", "hit_distance_sc", "launch_speed", "launch_angle", "launch_speed_angle", "estimated_ba_using_speedangle", "estimated_woba_using_speedangle", "estimated_slg_using_speedangle", "babip_value", "iso_value", "woba_value", "woba_denom"]
swing = ["bat_speed", "swing_length", "miss_distance", "hyper_speed", "attack_angle", "attack_direction", "swing_path_tilt", "intercept_ball_minus_batter_pos_x_inches", "intercept_ball_minus_batter_pos_y_inches"]
fielders = [f"fielder_{n}" for n in range(1, 10)]
prepitch_expectancy = ["home_win_exp", "bat_win_exp"]
postpitch_expectancy = ["delta_home_win_exp", "delta_run_exp", "delta_pitcher_run_exp"]
post_scores = ["post_away_score", "post_home_score", "post_bat_score", "post_fld_score"]
deprecated = ["spin_dir", "spin_rate_deprecated", "break_angle_deprecated", "break_length_deprecated", "tfs_deprecated", "tfs_zulu_deprecated", "umpire", "sv_id", "age_pit_legacy", "age_bat_legacy"]

In [ ]:
total = throws_df.count()
(throws_df.groupBy(["pitch_name", "pitch_type"])
    .count()
    .withColumn("%", F.round(F.col("count") / total * 100, 2))
    .orderBy("count", ascending=False)
    .show())

In [ ]:
# confronta le distribuzioni dei tiri tra tutte le stagioni disponibili
window = W.partitionBy("game_year")
(throws_df.groupby(["game_year", "pitch_name"])
    .count()
    .withColumn("year_count", F.sum("count").over(window))
    .withColumn("%", F.round(F.col("count") / F.col("year_count") * 100, 2))
    .withColumn("rank", F.row_number().over(window.orderBy(F.desc("count"))))
    .filter(F.col("rank") <= 5)
    .select(["game_year", "pitch_name", "%", "rank"])
).show(100, truncate=False)

In [ ]:
throws_df = throws_df.withColumn(
    "bs_count",
    F.concat(F.col("balls"), F.lit("-"), F.col("strikes"))
)

pitch_counts = (
    throws_df
    .groupBy("bs_count", "pitch_name")
    .count()
)

total_window = W.partitionBy("bs_count")
rank_window = W.partitionBy("bs_count").orderBy(F.desc("count"))

pitch_counts = (
    pitch_counts
    .withColumn("total", F.sum("count").over(total_window))
    .withColumn("%", F.round(F.col("count") / F.col("total") * 100, 2))
    .withColumn("rank", F.row_number().over(rank_window))
    .filter(F.col("rank") <= 5)
    .orderBy("bs_count", "rank")
)

pitch_counts.show(1000, truncate=False)

La cosa più evidente è il 3-0: la 4-Seam Fastball passa dal ~28–33% tipico di molti count al 60,82%, e insieme al Sinker arriva all'83,31%. È esattamente il tipo di relazione che speravamo di trovare nelle feature pre-pitch.
Si vedono anche altri pattern interessanti. Nei pitcher-friendly counts compaiono maggiormente breaking/off-speed pitch: per esempio Sweeper entra nella top 5 in 0-2 (10,28%), 1-2 (9,64%) e 2-2 (8,41%), mentre scompare dai count più favorevoli al battitore. Il Changeup passa dal non essere nemmeno nella top 5 a 0-0 all'11–14% circa in diversi count con strike. Al contrario, a 2-0 Fastball + Sinker rappresentano quasi il 59% dei lanci.


In [ ]:
window = W.partitionBy(["stand", "p_throws"])

(throws_df.groupBy(["stand", "p_throws", "pitch_name"])
.count()
.withColumn("tot", F.sum(F.col("count")).over(window))
.withColumn("%", F.round(F.col("count") / F.col("tot") * 100, 2))
.withColumn("rank", F.row_number().over(window.orderBy(F.desc("count"))))
.filter(F.col("rank") <= 5)
.orderBy(["stand", "p_throws", "rank"])
.show(100))

il changeup sembra fortemente associato ai matchup opposite-handed. 
Anche Slider/Sweeper mostrano quasi il comportamento opposto.

In [ ]:
window = W.partitionBy(["player_name"])
(throws_df.groupby(["player_name", "pitch_name"])
.count()
.withColumn("pitch_per_player", F.sum("count").over(window))
.withColumn("pitch%", F.round(F.col("count") / F.col("pitch_per_player") * 100, 2))
.withColumn("rank", F.row_number().over(window.orderBy("pitch%")))
.orderBy(F.col("pitch_per_player").desc(), F.col("rank").desc())
.select(["player_name", "pitch_name", "pitch%", "rank"])
.show(10000))

In [ ]:
throws_df.select("events").distinct().show()

In [ ]:
throws_df.select("description").distinct().show(truncate=False)